In [ ]:
import os
import re
import pandas as pd
import torch
from sqlalchemy import create_engine, inspect
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Database connection details
rds_host = os.getenv('rds_host')
username = os.getenv('rds_username')
training_data_table_name = os.getenv('training_data_table_name')
password = os.getenv('password')
database_name = os.getenv('database_name')
port = os.getenv('port')

# Establish database connection
connection_string = f'postgresql+psycopg2://{username}:{password}@{rds_host}:{port}/{database_name}'
engine = create_engine(connection_string)

# Function to get the latest version of the table
def get_latest_version_table(engine, base_table_name):
    inspector = inspect(engine)
    tables = inspector.get_table_names()
    
    versioned_tables = []
    pattern = re.compile(f"^{base_table_name}V(\d+)$")  # Matches "tableNameV1", "tableNameV2", etc.

    for table in tables:
        match = pattern.match(table)
        if match:
            versioned_tables.append((table, int(match.group(1))))

    if versioned_tables:
        latest_table = max(versioned_tables, key=lambda x: x[1])[0]  # Get table with highest version number
        return latest_table
    else:
        return base_table_name  # If no versioned tables exist, fallback to base name

# Get the latest table version
latest_table_name = get_latest_version_table(engine, training_data_table_name)
print(f"Using latest table: {latest_table_name}")

# Read data from the latest version of the table
df = pd.read_sql(f'SELECT * FROM "{latest_table_name}"', con=engine)

# Define target and features
target_col = 'Weekly_Sales'
features = df.drop(columns=[target_col])  # Remove target from features
target = df[target_col]

# Convert boolean columns to integers (True → 1, False → 0)
features = features.astype({col: 'int' for col in features.select_dtypes(include=['bool']).columns})

# Ensure all features are numeric
features = features.apply(pd.to_numeric, errors='coerce')

# Fill NaN values with mean (alternative: use median or drop rows)
features.dropna()

# Train-test split (80-20)
Train_X, Test_X, Train_y, Test_y = train_test_split(features, target, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
Train_X_tensor = torch.tensor(Train_X.values, dtype=torch.float32)
Test_X_tensor = torch.tensor(Test_X.values, dtype=torch.float32)
Train_y_tensor = torch.tensor(Train_y.values, dtype=torch.float32).unsqueeze(1)  # Ensure shape [N, 1]
Test_y_tensor = torch.tensor(Test_y.values, dtype=torch.float32).unsqueeze(1)

# Compute mean and std from training set (for standardization)
mean = Train_X_tensor.mean(dim=0, keepdim=True)
std = Train_X_tensor.std(dim=0, keepdim=True)

# Standardization
Train_X_std = (Train_X_tensor - mean) / std
Test_X_std = (Test_X_tensor - mean) / std  # Use training mean/std for test data

# Convert back to DataFrame for visualization
Train_X_std_df = pd.DataFrame(Train_X_std.numpy(), columns=features.columns)
Test_X_std_df = pd.DataFrame(Test_X_std.numpy(), columns=features.columns)

# Display summary statistics
print('\033[1mStandardization on Training set'.center(120))
display(Train_X_std_df.describe())

print('\n','\033[1mStandardization on Testing set'.center(120))
display(Test_X_std_df.describe())

# Print tensor shapes
print(f"Train X shape: {Train_X_std.shape}, Train y shape: {Train_y_tensor.shape}")
print(f"Test X shape: {Test_X_std.shape}, Test y shape: {Test_y_tensor.shape}")


Using latest table: trainingdataV1
                                          Standardization on Training set                                           


,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,weekday,year_2011,year_2012,month_2,month_3,...,Store_36,Store_37,Store_38,Store_39,Store_40,Store_41,Store_42,Store_43,Store_44,Store_45
count,1.898000e+03,1.898000e+03,1.898000e+03,1.898000e+03,1.898000e+03,0.0,1.898000e+03,1.898000e+03,1.898000e+03,1.898000e+03,...,1.898000e+03,1.898000e+03,1.898000e+03,1.898000e+03,1.898000e+03,1.898000e+03,1.898000e+03,1.898000e+03,1.898000e+03,1.898000e+03
mean,2.411821e-08,1.065221e-07,3.416747e-07,-3.778520e-07,4.421672e-08,NaN,5.024627e-10,-2.512314e-08,3.014777e-09,-3.215762e-08,...,4.019702e-09,-1.507388e-08,-1.105418e-08,5.024627e-09,5.024627e-08,2.813792e-08,2.411821e-08,2.034974e-08,3.517239e-08,1.205911e-08
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,9.999999e-01,NaN,9.999999e-01,9.999999e-01,1.000000e+00,9.999999e-01,...,9.999999e-01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,9.999999e-01,9.999999e-01,1.000000e+00,9.999999e-01
min,-2.755425e-01,-2.977864e+00,-1.833827e+00,-1.266915e+00,-2.718287e+00,NaN,-7.633339e-01,-6.141332e-01,-3.043677e-01,-3.074560e-01,...,-1.644446e-01,-1.627478e-01,-4.594366e-02,-1.428963e-01,-1.309196e-01,-1.593057e-01,-1.485496e-01,-1.485496e-01,-1.503908e-01,-1.485496e-01
25%,-2.755425e-01,-7.344463e-01,-9.765335e-01,-1.092731e+00,-6.700782e-01,NaN,-7.633339e-01,-6.141332e-01,-3.043677e-01,-3.074560e-01,...,-1.644446e-01,-1.627478e-01,-4.594366e-02,-1.428963e-01,-1.309196e-01,-1.593057e-01,-1.485496e-01,-1.485496e-01,-1.503908e-01,-1.485496e-01
50%,-2.755425e-01,1.068522e-01,1.161373e-01,3.907361e-01,9.831084e-02,NaN,-7.633339e-01,-6.141332e-01,-3.043677e-01,-3.074560e-01,...,-1.644446e-01,-1.627478e-01,-4.594366e-02,-1.428963e-01,-1.309196e-01,-1.593057e-01,-1.485496e-01,-1.485496e-01,-1.503908e-01,-1.485496e-01
75%,-2.755425e-01,7.861806e-01,8.287485e-01,9.845363e-01,6.337793e-01,NaN,1.309352e+00,1.627453e+00,-3.043677e-01,-3.074560e-01,...,-1.644446e-01,-1.627478e-01,-4.594366e-02,-1.428963e-01,-1.309196e-01,-1.593057e-01,-1.485496e-01,-1.485496e-01,-1.503908e-01,-1.485496e-01
max,3.627291e+00,2.125806e+00,2.476392e+00,1.312112e+00,2.674527e+00,NaN,1.309352e+00,1.627453e+00,3.283769e+00,3.250784e+00,...,6.077872e+00,6.141239e+00,2.175432e+01,6.994396e+00,7.634250e+00,6.273932e+00,6.728210e+00,6.728210e+00,6.645841e+00,6.728210e+00



                                            Standardization on Testing set                                           


,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,weekday,year_2011,year_2012,month_2,month_3,...,Store_36,Store_37,Store_38,Store_39,Store_40,Store_41,Store_42,Store_43,Store_44,Store_45
count,475.000000,475.000000,475.000000,475.000000,475.000000,0.0,475.000000,475.000000,475.000000,475.000000,...,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000,475.000000
mean,0.028468,-0.013710,0.054887,-0.013533,-0.028805,NaN,0.039559,0.027668,-0.024871,-0.015306,...,-0.059311,-0.069847,-0.045944,0.067466,0.016210,-0.023869,0.025179,0.025179,0.049919,-0.003776
std,1.047085,0.982760,0.990847,1.001633,1.008202,NaN,1.010772,1.014352,0.962655,0.977856,...,0.804106,0.760416,0.000000,1.208404,1.059813,0.924528,1.080262,1.080262,1.150659,0.988267
min,-0.275543,-2.724111,-1.743131,-1.266915,-2.718287,NaN,-0.763334,-0.614133,-0.304368,-0.307456,...,-0.164445,-0.162748,-0.045944,-0.142896,-0.130920,-0.159306,-0.148550,-0.148550,-0.150391,-0.148550
25%,-0.275543,-0.708126,-0.881519,-1.094398,-0.689143,NaN,-0.763334,-0.614133,-0.304368,-0.307456,...,-0.164445,-0.162748,-0.045944,-0.142896,-0.130920,-0.159306,-0.148550,-0.148550,-0.150391,-0.148550
50%,-0.275543,0.092545,0.198196,0.355569,0.044432,NaN,-0.763334,-0.614133,-0.304368,-0.307456,...,-0.164445,-0.162748,-0.045944,-0.142896,-0.130920,-0.159306,-0.148550,-0.148550,-0.150391,-0.148550
75%,-0.275543,0.733945,0.848183,0.990435,0.616787,NaN,1.309352,1.627453,-0.304368,-0.307456,...,-0.164445,-0.162748,-0.045944,-0.142896,-0.130920,-0.159306,-0.148550,-0.148550,-0.150391,-0.148550
max,3.627291,1.995691,2.130884,1.304515,2.674527,NaN,1.309352,1.627453,3.283769,3.250784,...,6.077872,6.141239,-0.045944,6.994396,7.634250,6.273932,6.728210,6.728210,6.645841,6.728210


Train X shape: torch.Size([1898, 63]), Train y shape: torch.Size([1898, 1])
Test X shape: torch.Size([475, 63]), Test y shape: torch.Size([475, 1])
